In [6]:

from langchain_community.vectorstores import Chroma
import os

from langchain_text_splitters import RecursiveCharacterTextSplitter
sample_text = """
Retrieval-Augmented Generation (RAG) is a technique that bridges the gap between 
a large language model's general knowledge and your specific, private data. 
By using local models like Llama 3.1 and local embeddings, you ensure that 
sensitive information never leaves your machine. The first step in any RAG 
pipeline is data ingestion, which involves loading documents and splitting 
them into chunks. Proper chunking is critical because if chunks are too small, 
they lose semantic meaning. If they are too large, they might exceed the 
model's context window or dilute the specific answer you are looking for.
""" * 10 

with open("local_rag_guide.txt", "w") as file:
    file.write(sample_text)

from langchain_community.document_loaders import TextLoader
loader = TextLoader("local_rag_guide.txt")
docs = loader.load()

print(f"Successfully loaded {len(docs)} document(s).")
print(f"Total character count: {len(docs[0].page_content)}")

small_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
medium_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
large_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

small_chunks = small_splitter.split_documents(docs)
medium_chunks = medium_splitter.split_documents(docs)
large_chunks = large_splitter.split_documents(docs)

from langchain_community.embeddings import OllamaEmbeddings

ollama_embeddings = OllamaEmbeddings(model="nomic-embed-text")


persist_directory = "./chroma_db"

print("Creating vector database. This might take a moment...")

vector_db = Chroma.from_documents(
    documents=medium_chunks, 
    embedding=ollama_embeddings, 
    persist_directory=persist_directory 
)

print(f"Success! Database created and saved to {persist_directory}")

Successfully loaded 1 document(s).
Total character count: 6160
Creating vector database. This might take a moment...
Success! Database created and saved to ./chroma_db


Basic vector search (what we just did in Lesson 3) is incredibly fast. It uses what's called a Bi-Encoder. It embeds the question, embeds the chunks, and just compares their coordinates. However, because it processes them separately, it can sometimes miss the deep semantic relationship between the question and the answer. It's fast, but a little bit "dumb."

To fix this, we introduce a Cross-Encoder (the Reranker). A Cross-Encoder is a heavier, smarter model. Instead of looking at the question and the chunk separately, it feeds them into the neural network together and asks the model: "On a scale of low to high, how perfectly does this specific chunk answer this specific question?"

Because Cross-Encoders are heavy and slow, we don't use them on the whole database. We use a two-stage retrieval pipeline:
1. Stage 1 (The Intern): We use our fast ChromaDB vector search to cast a wide net and pull the top 10 to 15 chunks.
2. Stage 2 (The Expert): We hand those 15 chunks to our Cross-Encoder. It carefully reads them alongside the query, re-scores them, and bubbles the true top 3 to the very top.

In [8]:
# 1. Broad Retrieval: Cast a wider net
# We fetch 10 results instead of 3 using our original query
query = "What happens if my chunks are too small or too large?"
broad_results = vector_db.similarity_search(query, k=10)
print(f"Retrieved {len(broad_results)} broad results from ChromaDB.")

# 2. Load the Cross-Encoder Reranker from HuggingFace
# We are using BAAI's bge-reranker-base, a state-of-the-art open-source reranker
from sentence_transformers import CrossEncoder

print("Downloading/Loading Reranker Model... this might take a minute...")
reranker = CrossEncoder("BAAI/bge-reranker-base")

# 3. Prepare the data pairs for the reranker: [ [query, chunk1], [query, chunk2], ... ]
pairs = [[query, doc.page_content] for doc in broad_results]

# 4. Re-score the pairs using the heavier Cross-Encoder model
print("Reranking chunks... ")
new_scores = reranker.predict(pairs)

# 5. Combine the documents with their new scores and sort them
# Note: Unlike Chroma's "distance" (lower is better), Reranker scores are "logits" (higher is better!)
scored_docs = zip(broad_results, new_scores)
sorted_docs = sorted(scored_docs, key=lambda x: x[1], reverse=True)

# 6. Show the refined Top 3!
print("\n" + "="*50)
print("--- NEW TOP 3 AFTER RERANKING ---")
for i, (doc, score) in enumerate(sorted_docs[:3]):
    print(f"Rank {i+1} | New Reranker Score: {score:.4f}")
    # We will just print the first 100 characters so it doesn't flood your screen
    print(f"Content: {doc.page_content[:100]}...\n")

Retrieved 10 broad results from ChromaDB.
Downloading/Loading Reranker Model... this might take a minute...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 19592.72it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranking chunks... 

--- NEW TOP 3 AFTER RERANKING ---
Rank 1 | New Reranker Score: 0.7660
Content: they lose semantic meaning. If they are too large, they might exceed the 
model's context window or ...

Rank 2 | New Reranker Score: 0.7660
Content: they lose semantic meaning. If they are too large, they might exceed the 
model's context window or ...

Rank 3 | New Reranker Score: 0.7660
Content: they lose semantic meaning. If they are too large, they might exceed the 
model's context window or ...



How does the reranker actually work? **bi-encoder vs cross-encoder**

* Because of its pre-training, nomic-embed-text naturally places the mathematical vector for "Apple" very close to the vector for "iPhone" in its 768-dimensional space. So, even if your query says "Apple" and the chunk only says "iPhone", the database will still recognize them as a decent match.
    * However, because it's just comparing two isolated summaries (the "smoothies"), the connection might be weak if the chunk also talks about a lot of other random things.
    * This is where the Cross-Encoder flexes its muscles. Because it processes the query and the chunk together, its self-attention mechanism acts like a spotlight.
        * When it reads [Query] Where is the Apple store? [Chunk] The new iPhone is available downtown

When you use ChromaDB with nomic-embed-text, you are using a Bi-Encoder.
* The Process: It processes the User Query in completely isolated Room A. It processes the Document Chunk in completely isolated Room B.
* The Output: They each output a vector (a summary of numbers). ChromaDB just measures the physical distance between those two summaries.
* The Flaw: Because the query and the chunk are processed in complete isolation, they never get to "talk" to each other. 
    * If your query is "Where is the Apple store?" 
    * and the chunk says "The new iPhone is available downtown"
    * Then a Bi-Encoder might miss the connection because the word "Apple" and "iPhone" were processed in separate rooms


A Cross-Encoder does not process them in separate rooms.
* The Process: It takes the User Query and the Document Chunk, glues them together with a special separator token, and feeds them into the neural network at the exact same time.
    * (e.g., [CLS] Where is the Apple store? [SEP] The new iPhone is available downtown [SEP])
* The Magic (Self-Attention): Because they are inside the Transformer model together, every single word in the query gets to mathematically "look at" every single word in the chunk. 
    * The word "Apple" in the query can directly interact with the word "iPhone" in the chunk. The model instantly recognizes the semantic relationship between the two concepts in real-time.
* The Output: A Cross-Encoder does not output a vector. Instead, it outputs one single number (a classification score, or "logit") that represents exactly how relevant the chunk is to the query.

**the difference is that with bi-encoding you can precalculate everything such that you judt try match distances, but you can't do that with a cross encoder - so you can't possibly try and loop through 10,000 chunks in cross-encoders, but you can do that pre-emptively in a bi-encoder**


